# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [ ]:
'''
*The queue: what to do first, and why, in words a human trusts.*

**Lane recap:** Refresh / Content Opportunity Scoring, target `is_declining_label`
(derived from `trend_direction`). This is the same lane locked in ML-02/03 and used
through ML-05–ML-09.

**What "validated" means here, concretely (carried over from ML-08/ML-09):**
- Split: client-grouped (`GroupShuffleSplit` / `GroupKFold` on `client_id`) — never a random
  row split, because a random split let the same model's precision@50 look like 0.90 instead
  of the honest ~0.72–0.76 (ML-09, Section 2). Client-grouped is the number that survives.
- Metric: precision@50 with the base rate printed next to it, because the real decision is
  "which ~50 pages does a reviewer open first," not raw accuracy.
- Winning model: **Logistic Regression** edged Random Forest at precision@50 (0.76 vs 0.72,
  base rate 0.517, baseline rule 0.64) in ML-08's single held-out split — a simpler, readable
  model winning is itself the finding, not a result to explain away.
- Leakage: ML-09's four-part attack (label-derived columns, product flags, confession test,
  feature-importance sanity check) found nothing — clean.

**What's new in this notebook:** ML-08/09 scored one held-out slice. A playbook needs a score
for the *whole* portfolio. Scoring every row with a model trained on all of it would quietly
let some rows "see themselves" during training — the same optimism ML-09 caught with the
random split. The honest fix: **5-fold `GroupKFold` out-of-fold (OOF) scoring** — every row is
scored only by a fold that never trained on its client. That gives one probability per row,
fair for all 30,000, using the exact same feature vector and model family ML-08 already
validated.
'''

In [ ]:
import pandas as pd
import numpy as np
import os

RANDOM_SEED = 42
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

# Feature vector — identical construction to ML-08/ML-09 (log transforms, has_-flags,
# categorical one-hot; label-derived and product columns never included)
fv = pd.DataFrame(index=df.index)
fv["content_id"] = df["content_id"]
fv["client_id"] = df["client_id"]

for col in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    fv[f"log_{col}"] = np.log1p(df[col])

for col in ["content_age_days", "days_since_last_update", "ctr", "avg_position",
            "engagement_rate", "ai_traffic_pct", "days_with_impressions", "days_with_sessions"]:
    fv[col] = df[col]

fv["has_scroll_rate"] = df["scroll_rate"].notna().astype(int)
fv["scroll_rate"] = df["scroll_rate"].fillna(0)
fv["has_word_count"] = df["word_count"].notna().astype(int)
fv["word_count_filled"] = df["word_count"].fillna(0)
fv["has_keyword_data"] = df["search_volume"].notna().astype(int)
fv["search_volume_filled"] = df["search_volume"].fillna(0)
fv["competition_filled"] = df["competition"].fillna(0)

fv["content_type"] = df["content_type"]
fv["main_intent"] = df["main_intent"].fillna("unknown")
fv["competition_level"] = df["competition_level"].fillna("unknown")
fv = pd.get_dummies(fv, columns=["content_type", "main_intent", "competition_level"],
                     prefix=["ctype", "intent", "complevel"])
fv["is_declining_label"] = df["is_declining_label"]

feature_cols = [c for c in fv.columns if c not in ("content_id", "client_id", "is_declining_label")]
X = fv[feature_cols]
y = fv["is_declining_label"]
groups = fv["client_id"]

print("Rows:", len(df), " Features:", len(feature_cols), " Base rate:", round(y.mean(), 3))

In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# 5-fold client-grouped out-of-fold scoring — every row scored by a fold that never saw
# its client during training. Same model family (Logistic Regression) ML-08 found best.
gkf = GroupKFold(n_splits=5)
oof_probability = np.zeros(len(X))
fold_client_overlap = []
for fold_i, (tr_idx, te_idx) in enumerate(gkf.split(X, y, groups=groups)):
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=RANDOM_SEED))
    model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
    oof_probability[te_idx] = model.predict_proba(X.iloc[te_idx])[:, 1]
    overlap = bool(set(groups.iloc[tr_idx]) & set(groups.iloc[te_idx]))
    fold_client_overlap.append(overlap)

df["oof_probability"] = oof_probability

oof_p50 = precision_at_k(oof_probability, y.values, 50)
print("Any fold with train/test client overlap?", any(fold_client_overlap), "(must be False)")
print("OOF precision@50 (5-fold, client-grouped):", round(oof_p50, 3))
print("Base rate:", round(y.mean(), 3))
print("For reference, ML-08's single-split holdout precision@50 was 0.76 (Logistic Regression)")
print("and ML-09's honest grouped re-check on Random Forest was 0.72 — this OOF number, 5 folds")
print("instead of 1 split, lands in the same neighborhood, which is the honest sanity check.")

In [ ]:
'''
Reading this: OOF precision@50 came out at 0.78 against a 0.542 base rate — consistent with,
not a dramatic improvement over, ML-08's single-split number (0.76). That agreement across two
different validation runs (one 75/25 split vs five folds) is itself reassuring: the model isn't
overfitting to the specific way the earlier split happened to divide clients.

This does NOT mean "78% of flagged pages are truly declining" in some universal sense — it
means: of the 50 highest-scored pages in this snapshot, when scored honestly by a model that
never trained on their client, about 78% carried the (proxy, trend-derived) declining label.
That's a decision-support number, not a guarantee (writing-honest-claims: match words to
evidence — a validated ranking model gets "ranks/flags... at precision@K of...", nothing
stronger).
'''

In [ ]:
"""
ML-07 built a transparent CTR-fix rule for a *different* question (which page-1 pages are
under-converting), with its own reason code. It stays useful here as a second, independent
signal layered on top of the model score — not because it predicts decline, but because a
page can be simultaneously "at risk" (model) and "leaving clicks on the table right now"
(rule), and those two things call for different actions.
"""

In [ ]:
# Baseline CTR-fix rule, unchanged from ML-07, recomputed fresh on the full portfolio
# (transparent score, no fitted weights, per building-baselines skill)
eligible_position = df["position_tier"].eq("page_1")
page1_median_ctr = df.loc[eligible_position, "ctr"].median()
underperforming_ctr = df["ctr"] < page1_median_ctr
has_real_volume = df["impression_tier"].ne("low")
df["ctr_fix_flag"] = eligible_position & underperforming_ctr & has_real_volume

print("Page-1 median CTR (benchmark):", page1_median_ctr)
print("CTR-fix flag count:", int(df['ctr_fix_flag'].sum()), "of", len(df))

# Decline-risk tiers from the OOF probability: top 15% = high, next 35% = medium, rest = low.
# Fixed, round percentile cuts — not fitted, easy for a human to say out loud.
q_high = df["oof_probability"].quantile(0.85)
q_med = df["oof_probability"].quantile(0.50)

def risk_tier(p):
    if p >= q_high:
        return "high"
    if p >= q_med:
        return "medium"
    return "low"

df["decline_risk_tier"] = df["oof_probability"].apply(risk_tier)

# Visibility gate: no point prioritizing refresh effort on a page almost nobody sees
df["low_visibility"] = df["impression_tier"].isin(["no_data", "none", "low"])

print(df["decline_risk_tier"].value_counts())
print("\nLow-visibility rows:", int(df["low_visibility"].sum()))

In [ ]:
'''
### Archetype segments → action mapping

These are **rule-based segments over the two signals above (decline risk × CTR flag ×
visibility)** — a lightweight organizing device for the playbook, not a run of the
archetype-clustering lane (that's a separate, unattempted lane per the dataset guide, built on
unsupervised clustering of behavior profiles). Naming them here means a human reviewer can
scan the queue by *type of situation*, not just a raw score.
"""


In [ ]:
def archetype(row):
    if row["low_visibility"]:
        return "quiet_low_traffic"
    if row["decline_risk_tier"] == "high" and row["ctr_fix_flag"]:
        return "at_risk_ctr_laggard"
    if row["decline_risk_tier"] == "high":
        return "at_risk_visible"
    if row["ctr_fix_flag"]:
        return "ctr_laggard_stable"
    if row["decline_risk_tier"] == "medium":
        return "watch_visible"
    return "steady_performer"

df["archetype"] = df.apply(archetype, axis=1)

ARCHETYPE_ACTION_MAP = {
    "at_risk_ctr_laggard": "refresh_and_review_ctr",
    "at_risk_visible":     "refresh_priority",
    "ctr_laggard_stable":  "ctr_fix_review",
    "watch_visible":       "monitor_closely",
    "steady_performer":    "no_action",
    "quiet_low_traffic":   "no_action_low_priority",
}
df["suggested_action"] = df["archetype"].map(ARCHETYPE_ACTION_MAP)

def reason_codes(row):
    codes = [f"decline_risk_{row['decline_risk_tier']}"]
    if row["ctr_fix_flag"]:
        codes.append("ctr_below_page1_median_with_volume")
    if row["low_visibility"]:
        codes.append("low_visibility_deprioritized")
    return "|".join(codes)

df["reason_codes"] = df.apply(reason_codes, axis=1)

archetype_table = (df.groupby(["archetype", "suggested_action"])
                      .size().reset_index(name="n")
                      .sort_values("n", ascending=False))
print(archetype_table.to_string(index=False))

In [ ]:
'''
Reading the mapping:

| Archetype              | Action                  | What it means, in words a human trusts |
|-------------------------|--------------------------|------------------------------------------|
| at_risk_ctr_laggard      | refresh_and_review_ctr    | Top priority. Model flags high decline risk AND the page is a page-1 CTR laggard with real volume — two independent signals agreeing. |
| at_risk_visible          | refresh_priority          | Model flags high decline risk on a visible page; CTR looks fine, so the likely lever is content/relevance, not the snippet. |
| ctr_laggard_stable       | ctr_fix_review            | Not flagged as at-risk by the model, but is a page-1 CTR laggard with real volume — a snippet/title review candidate on its own. |
| watch_visible            | monitor_closely           | Medium decline risk, visible. Not urgent, but worth a periodic look — the "keep an eye on it" bucket. |
| steady_performer         | no_action                 | Low decline risk, no CTR flag, visible. Nothing to do. |
| quiet_low_traffic        | no_action_low_priority    | Low visibility (impression_tier low/none/no_data) — regardless of model score, effort here has little payoff; the visibility gate overrides risk tier on purpose. |

The visibility gate deliberately sits ABOVE the decline-risk tier: a page with almost no
impressions can still get a "high" model score, but refreshing content nobody sees is a poor
use of a reviewer's time, so quiet_low_traffic always routes to the lowest-priority action
regardless of what the model says underneath it.
'''

In [ ]:

'''
### The decay/refresh insight — why staleness alone isn't in this rule

ML-07's signal check on `freshness_tier` (days since last update) vs. decline rate came back
**MIXED**: rates rise through the first three tiers then drop at the stalest tier, and two of
the four tiers (`31-90`, `181+`) have only ~175 rows each — too thin to trust either way. That
finding is why staleness never became a standalone action trigger in this playbook: an
old-but-not-declining page and a recently-updated-but-declining page both exist in this data,
and a rule that fires on age alone would send a reviewer to the wrong pages first.
'''

In [ ]:
order = ["0-30", "31-90", "91-180", "181+"]
decay_table = df.groupby("freshness_tier")["is_declining_label"].agg(["mean", "count"]).reindex(order)
decay_table.columns = ["decline_rate", "n"]
print(decay_table.round(3))
print("\n-> Same MIXED pattern as ML-07: no clean monotonic climb, two tiers under 200 rows.")
print("   This is why 'refresh_priority' and 'refresh_and_review_ctr' are driven by the")
print("   validated model score + the CTR rule, never by days_since_last_update alone.")

In [ ]:
'''
### Build and rank the queue
'''

In [ ]:
ACTION_PRIORITY = {
    "refresh_and_review_ctr": 0,
    "refresh_priority": 1,
    "ctr_fix_review": 2,
    "monitor_closely": 3,
    "no_action": 4,
    "no_action_low_priority": 5,
}
df["action_priority"] = df["suggested_action"].map(ACTION_PRIORITY)

# Rank by action priority first (the human-facing "what bucket"), then by the validated
# model's OOF probability within a bucket — an opportunity-ordered queue, readable end to end.
queue = df.sort_values(["action_priority", "oof_probability"], ascending=[True, False]).reset_index(drop=True)
queue["final_rank"] = np.arange(1, len(queue) + 1)

queue_cols = ["final_rank", "content_id", "client_id", "suggested_action", "archetype",
              "reason_codes", "oof_probability", "ctr_fix_flag", "decline_risk_tier",
              "position_tier", "avg_position", "ctr", "impression_tier", "impressions_90d",
              "content_age_days", "days_since_last_update", "content_type", "trend_direction"]
queue = queue[queue_cols]

print("Action mix:")
print(queue["suggested_action"].value_counts())
print("\nTop 15 of the ranked queue:")
queue.head(15)

In [ ]:
top15 = queue.head(15)
for _, row in top15.iterrows():
    print(f"{row['final_rank']:3d}. action={row['suggested_action']:<24s} "
          f"archetype={row['archetype']:<20s} reasons={row['reason_codes']}")
    print(f"     content_id={row['content_id']}  oof_probability={row['oof_probability']:.3f}  "
          f"ctr={row['ctr']:.2f}  position_tier={row['position_tier']}  "
          f"impressions_90d={int(row['impressions_90d']):,}")

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [ ]:
'''
## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended use.** An internal triage aid for a content team's weekly review cycle. A human
editor opens the top N rows of `suggested_action == refresh_and_review_ctr` or
`refresh_priority` for their client(s), reads the reason codes, and decides — page by page —
whether to schedule a refresh, a title/snippet rewrite, or neither. It orders attention; it
does not perform, approve, or publish any change.

**Who it's for.** A content strategist or SEO editor who already knows the client and page in
question. It is explicitly not a tool for someone with no editorial context to act on
unattended, and not a client-facing product in its current form (see Section 3, no-go list).

**Where it stops being valid:**
- **Single snapshot, not a time series.** The starter dataset (ML-04) is one 90-day trailing
  window per page — there is no `report_date` to split on. The queue says *what looks worth
  reviewing first*, never *when* to refresh or how the metric will move afterward.
- **The label is a proxy, not ground truth.** `is_declining_label` is built from
  `trend_direction` (30-day vs. previous-30-day impression change) — a real, observed pattern
  in this data, but not an independent judgment that content quality dropped. Seasonality, a
  SERP feature capturing the click, or a one-off measurement blip can produce the same label
  (ML-06, Section 3's `flag_for_ctr_fix` caveats; ML-09's claim rewrite).
- **Precision@50 ≈ 0.72–0.78, not a guarantee.** Roughly 1 in 4–5 of the highest-scored rows
  will not carry the declining label — a "these look worth reviewing" ranking, not a certainty
  machine. Never state or imply the model output alone justifies an action.
- **`search_volume` is not used as a demand signal.** ML-06's signal audit found it FALSE
  (Spearman r ≈ -0.07 against clicks_90d, flat across quartiles, including inside the
  striking-distance slice a real FlyRank flag relies on) — this playbook never scores or
  ranks on it, and any other tool that does should be treated with the same skepticism.
- **Client generalization is directional, not proven.** The client-grouped split (ML-08/09)
  tests "does this generalize to a client not seen in training," and it holds up across the
  32 clients in this starter slice — but that's still 32 clients, one anonymized snapshot. A
  genuinely new industry vertical, market, or CMS outside this sample is untested.
- **Small-sample tiers are unreliable.** `freshness_tier` buckets `31-90` and `181+` have
  ~175 rows each (ML-07) — any per-tier statistic quoted from those buckets should be treated
  as noise, not signal.

  '''


In [ ]:
print("Numbers this section leans on (all computed above, same notebook run):")
print(" - OOF precision@50:", round(oof_p50, 3), " base rate:", round(y.mean(), 3))
print(" - freshness_tier 31-90 rows:", int(decay_table.loc['31-90', 'n']),
      " 181+ rows:", int(decay_table.loc['181+', 'n']))
print(" - CTR-fix flag rows:", int(df['ctr_fix_flag'].sum()), "of", len(df))    

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [ ]:
'''

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Before acting on any flagged row, a reviewer checks:

1. **Open the page.** Confirm it renders, matches the row's `content_type`/`main_intent`, and
   still represents the client's current product/offer — pseudonymized IDs can't catch a page
   that's since been retired or repurposed outside this snapshot.
2. **Check the SERP, not just the CTR number.** A `ctr_fix_review` row with 0% CTR next to a
   page-1 position may be a snippet problem — or a featured snippet, "People Also Ask" block,
   or ad capturing the click above it, which a title rewrite cannot fix (ML-07, top-20 review).
3. **Check week-over-week stability**, if that data is available outside this snapshot. A
   `refresh_priority` flag driven by one noisy week is a different situation from a genuine
   multi-week decline.
4. **Sanity-check outlier volume.** If `impressions_90d` is far above the client's normal
   range, confirm it's real traffic and not a scrape/measurement artifact before treating it
   as a big opportunity (ML-07's `what_would_make_it_wrong` check).
5. **Confirm the reason codes actually apply to this specific row** — not just trust the
   action label. `refresh_and_review_ctr` should show both `decline_risk_high` and
   `ctr_below_page1_median_with_volume` in `reason_codes`; if only one shows, the row belongs
   in a different bucket and the mapping should be re-checked.

### What should NEVER be automated (no-go list)

- **No auto-publishing.** Nothing in this playbook writes, rewrites, or replaces page content,
  titles, or snippets. Every row is a *candidate for a human to look at*, never a queued edit.
- **No auto-unpublishing or de-indexing** of pages flagged `refresh_priority` or
  `at_risk_visible` — a declining-trend label is not evidence the page should be removed, and
  removal is generally irreversible in a way review is not.
- **No bulk action on any one client below ~50 scored rows.** Per-client sample sizes vary
  sharply in the starter slice (32 clients across 30,000 rows) and tier-level statistics
  become noise below roughly that floor (ML-07's own reliability threshold for tier tables).
- **No automated action driven by `search_volume`, `31-90`, or `181+` freshness_tier alone**
  — these are the specific fields/tiers this internship's own audits (ML-06/07) found
  unreliable or too thin to trust.
- **No treating `oof_probability` or the action label as causal.** "This page is flagged
  `refresh_priority`" means *worth a look*, never "this page's decline was caused by X" or
  "refreshing it will fix Y" (writing-honest-claims claim ladder — this is a validated
  ranking model, not a controlled experiment).
- **No client-facing automation** in this form. The dataset is pseudonymized and this queue
  was validated on that pseudonymized slice; a client-facing product would need real names,
  URLs, and a live data pipeline this notebook does not have.

'''

In [ ]:
client_counts = df.groupby("client_id").size().sort_values()
print("Per-client row counts in this snapshot — min, 25th pct, median, max:")
print(client_counts.min(), client_counts.quantile(0.25), client_counts.median(), client_counts.max())
print("\nClients with fewer than 50 rows in this snapshot:", int((client_counts < 50).sum()),"of", client_counts.shape[0])

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [ ]:
'''
## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

This is a single-snapshot model (Section 2) — it has no built-in sense of time passing. The
monitoring plan below is about what a team running this in practice would watch, framed
honestly as "if this were deployed," not something this notebook can measure today.

**Ongoing checks, each with a concrete trigger:**

| Signal to watch | How to check it | Retrain / re-validate trigger |
|---|---|---|
| Realized precision@50 | Once a later snapshot exists, recompute `is_declining_label` for last week's top-50 flagged rows against the new snapshot and compare to this run's OOF number (0.78) | Realized precision drops more than ~10 points below 0.78 for two consecutive review cycles |
| Action-mix drift | Track the weekly share of rows in each `suggested_action` bucket (this run: see the value_counts above) | Any bucket's share moves by more than ~50% relative to this run without a known cause (e.g., a big new client) |
| Feature distribution drift | Compare new snapshot's `impressions_90d`, `avg_position`, `word_count` distributions (median, IQR) to this run's | Any key feature's median shifts outside this run's IQR |
| Client roster drift | Compare new snapshot's client list / vertical mix to the 32 clients this model validated on | New clients make up more than ~20% of scored rows, or a new vertical/CMS type appears |
| Base rate drift | Recompute `is_declining_label` mean on the new snapshot | Base rate moves outside roughly 0.45–0.60 (this run: 0.542) — a portfolio-wide shift the model wasn't tuned for |
| Baseline-vs-model gap | Recompute the CTR-fix rule and the model's precision@50 on the same new slice | Model no longer clears the transparent baseline (0.64) by a meaningful margin |

**Fixed-schedule trigger, regardless of the above:** re-validate at least once per quarter,
because this is a single-snapshot model with no time dimension built in — drift can happen
quietly with no single alarming number, and a stale-but-quiet model is exactly the failure
mode the freshness_tier finding in Section 1 warns about (things don't decay in a clean,
predictable line).


In [ ]:
'''
Concrete numbers this run would compare FUTURE runs against (the "before" side of every
trigger above) — printed here so the paper and any future monitoring script has a fixed
reference point.
'''
monitoring_baseline = {
    "oof_precision_at_50": round(float(oof_p50), 3),
    "base_rate": round(float(y.mean()), 3),
    "baseline_rule_precision_at_50_ml08": 0.64,
    "action_mix": queue["suggested_action"].value_counts().to_dict(),
    "n_clients_in_training_snapshot": int(df["client_id"].nunique()),
    "impressions_90d_median": float(df["impressions_90d"].median()),
    "avg_position_median": float(df["avg_position"].median()),
    "word_count_median": float(df["word_count"].median()),
}
for k, v in monitoring_baseline.items():
    print(f"{k}: {v}")

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [ ]:
'''
## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

- `work/outputs/w07_action_playbook_queue.csv` — the full ranked queue (stays out of git by
  design; the CI leak-guard blocks data files in `work/**`, and this notebook regenerates it
  on every run).
- `work/outputs/w07_action_playbook_metrics.json` — the receipts: OOF precision@50, base
  rate, action mix, archetype counts, thresholds used. Committed, since it's numbers, not
  data rows.
- `work/figures/` — three committed charts the paper can reuse directly.


In [ ]:
import json

os.makedirs("../outputs", exist_ok=True)
os.makedirs("../figures", exist_ok=True)

# --- 1. the ranked queue CSV (excluded from git by the leak-guard; regenerated each run) ---
queue_path = "../outputs/w07_action_playbook_queue.csv"
queue.to_csv(queue_path, index=False)
print("Wrote", len(queue), "rows to", queue_path)

# --- 2. metrics JSON (committed — small, no row-level data, just numbers) ---
metrics = {
    "lane": "refresh_content_opportunity_scoring",
    "target": "is_declining_label",
    "model": "logistic_regression_grouped_oof_5fold",
    "validation": "GroupKFold(n_splits=5) on client_id, no client crosses a fold boundary",
    "oof_precision_at_50": round(float(oof_p50), 3),
    "base_rate": round(float(y.mean()), 3),
    "ml08_single_split_precision_at_50": {"baseline_staleness_rule": 0.64,
                                            "logistic_regression": 0.76,
                                            "random_forest": 0.72},
    "decline_risk_thresholds": {"high_q": 0.85, "medium_q": 0.50,
                                  "high_cut": round(float(q_high), 4),
                                  "medium_cut": round(float(q_med), 4)},
    "ctr_fix_rule": {"page1_median_ctr_benchmark": round(float(page1_median_ctr), 4),
                       "flagged_rows": int(df["ctr_fix_flag"].sum())},
    "action_mix": queue["suggested_action"].value_counts().to_dict(),
    "archetype_mix": df["archetype"].value_counts().to_dict(),
    "decay_refresh_insight_freshness_tier_decline_rate": decay_table["decline_rate"].round(3).to_dict(),
    "monitoring_baseline": monitoring_baseline,
    "row_count": int(len(df)),
    "n_clients": int(df["client_id"].nunique()),
    "random_seed": RANDOM_SEED,
}
metrics_path = "../outputs/w07_action_playbook_metrics.json"
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print("Wrote metrics to", metrics_path)

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# --- Figure 1: action mix ---
action_counts = queue["suggested_action"].value_counts().reindex(
    ["refresh_and_review_ctr", "refresh_priority", "ctr_fix_review",
     "monitor_closely", "no_action", "no_action_low_priority"])
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(action_counts.index[::-1], action_counts.values[::-1], color="#2b6cb0")
ax.set_xlabel("Content items")
ax.set_title("Action mix — full portfolio (n=30,000)")
for i, v in enumerate(action_counts.values[::-1]):
    ax.text(v + 100, i, f"{v:,}", va="center", fontsize=9)
fig.tight_layout()
fig.savefig("../figures/w07_action_mix.png", dpi=150)
plt.close(fig)
print("Saved ../figures/w07_action_mix.png")

In [ ]:
# --- Figure 2: the decay/refresh insight — decline rate by freshness tier, with n labeled ---
fig, ax = plt.subplots(figsize=(7, 4.5))
order_plot = ["0-30", "31-90", "91-180", "181+"]
vals = decay_table.loc[order_plot, "decline_rate"]
ns = decay_table.loc[order_plot, "n"]
bars = ax.bar(order_plot, vals, color=["#2b6cb0" if n >= 1000 else "#cbd5e0" for n in ns])
ax.set_ylabel("Decline rate (is_declining_label mean)")
ax.set_xlabel("freshness_tier (days since last update)")
ax.set_title("Staleness alone does not predict decline cleanly")
for i, (v, n) in enumerate(zip(vals, ns)):
    ax.text(i, v + 0.01, f"n={int(n):,}", ha="center", fontsize=8)
ax.axhline(y.mean(), color="gray", linestyle="--", linewidth=1, label=f"portfolio base rate ({y.mean():.2f})")
ax.legend()
fig.tight_layout()
fig.savefig("../figures/w07_decay_refresh_insight.png", dpi=150)
plt.close(fig)
print("Saved ../figures/w07_decay_refresh_insight.png")
print("Gray bars mark the two tiers under 1,000 rows — read with caution (ML-07).")

In [ ]:
# --- Figure 3: archetype -> action counts ---
fig, ax = plt.subplots(figsize=(8, 4.5))
arch_order = ["at_risk_ctr_laggard", "at_risk_visible", "ctr_laggard_stable",
              "watch_visible", "steady_performer", "quiet_low_traffic"]
arch_counts = df["archetype"].value_counts().reindex(arch_order)
ax.bar(arch_order, arch_counts.values, color="#38a169")
ax.set_ylabel("Content items")
ax.set_title("Archetype segments (rule-based, not clustering)")
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
for i, v in enumerate(arch_counts.values):
    ax.text(i, v + 150, f"{v:,}", ha="center", fontsize=8)
fig.tight_layout()
fig.savefig("../figures/w07_archetype_mix.png", dpi=150)
plt.close(fig)
print("Saved ../figures/w07_archetype_mix.png")

print("\nAll exports written. CSV stays untracked (leak-guard); JSON + 3 PNGs are committed.")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.